<a href="https://colab.research.google.com/github/Shahad-Hossain/nasa-image-enhancement/blob/main/Homework_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mahotas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 29.5 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import numpy as np
import mahotas as mh
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

import glob
from PIL import Image

In [ ]:
# Fetching from an online hugging face dataset https://huggingface.co/datasets/hassanjbara/AI4MARS

print("Connecting to Hugging Face...")
ds = load_dataset("hassanjbara/AI4MARS", split="train", streaming=True)

features_list = []
labels_list = []

print("Extracting Haralick features from 5000 valid images...")
for example in ds:
    if example['label_mask'] is None:
        continue

    mask = np.array(example['label_mask'], dtype=np.int64).flatten()
    mask_filtered = mask[mask != 255]

    if len(mask_filtered) == 0:
        continue

    dominant_class = np.argmax(np.bincount(mask_filtered))

    img = np.array(example['image'].convert('L'))
    h_features = mh.features.haralick(img).mean(axis=0)
    selected = h_features[[0, 1, 2, 8]]

    features_list.append(selected)
    labels_list.append(dominant_class)

    if len(features_list) % 250 == 0:
        print(f"Done: {len(features_list)}/5000")

    if len(features_list) == 5000:
        break

X = np.array(features_list)
y = np.array(labels_list)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nReady! Data shape: {X_scaled.shape}")

Connecting to Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Extracting Haralick features from 5000 valid images...
Done: 250/5000
Done: 500/5000
Done: 750/5000


KeyboardInterrupt: 

In [ ]:
# 1. Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. Initialize and Train the Linear SVM
print("Training Linear SVM model...")
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train, y_train)

# 3. Predict on the test set
y_pred = svm_model.predict(X_test)

# 4. Print the results!
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Assuming 'svm_model' and 'scaler' from your previous code are still in memory!
class_names = {0: "Soil", 1: "Bedrock", 2: "Sand", 3: "Big Rock"}

def predict_custom_mars_data(image_paths, trained_model, fitted_scaler):
    valid_images = []
    features_list = []

    print(f"Processing {len(image_paths)} custom images...")

    for path in image_paths:
        try:
            # 1. Load the custom image and convert to grayscale
            img_pil = Image.open(path).convert('L')

            # Resize slightly if the custom images are massive (e.g., 4K NASA raw images)
            # to prevent mahotas from taking forever to compute the GLCM
            img_pil = img_pil.resize((512, 512))
            img_array = np.array(img_pil)

            # 2. Extract the exact same 4 Haralick features
            h_features = mh.features.haralick(img_array).mean(axis=0)
            selected = h_features[[0, 1, 2, 8]] # Energy, Contrast, Correlation, Entropy

            features_list.append(selected)
            valid_images.append(img_pil)

        except Exception as e:
            print(f"Could not process {path}: {e}")

    if not features_list:
        print("No valid images processed.")
        return

    # 3. Scale the features using the FITTED scaler
    # CRITICAL: Use transform(), NOT fit_transform()
    X_custom = np.array(features_list)
    X_custom_scaled = fitted_scaler.transform(X_custom)

    # 4. Ask the SVM what it thinks
    predictions = trained_model.predict(X_custom_scaled)

    # 5. Visualize the results
    num_images = len(valid_images)
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))

    # Handle single image case for plotting
    if num_images == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ax.imshow(valid_images[i], cmap='gray')
        pred_label = class_names.get(predictions[i], f"Class {predictions[i]}")
        ax.set_title(f"SVM Thinks:\n{pred_label}", fontsize=14, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Grab all JPG files from your custom folder
custom_image_paths = glob.glob("my_mars_test/*.jpg")

predict_custom_mars_data(custom_image_paths[:4], svm_model, scaler)